# 01 - Baseline reproductible

Charge le jeu synthetique, applique la **baseline** (classifieur d'image transparent, sans regle d'incertitude) sur tous les cas, puis affiche une sortie JSON complete et les metriques.

> Prototype pedagogique. Non destine au diagnostic. Validation par un professionnel qualifie requise.

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from src.inference import toy_predict
from src.guardrails import apply_safety_guardrails
from src.metrics import summarize_metrics

cases = pd.read_csv(ROOT / 'data' / 'synthetic_cases.csv')
cases.head()

In [ ]:
# Sortie JSON complete sur un cas d'exemple
sample = ROOT / 'data' / 'sample_images' / 'CXR_SYN_002_suspected_opacity.png'
pred = apply_safety_guardrails(toy_predict(sample, mode='baseline'))
print(json.dumps(pred, indent=2, ensure_ascii=False))

In [ ]:
# Baseline sur tous les cas + metriques
rows = []
for _, c in cases.iterrows():
    p = apply_safety_guardrails(toy_predict(ROOT / c['image_path'], mode='baseline'))
    rows.append({'case_id': c['case_id'], 'label': c['label'],
                 'predicted_class': p['predicted_class'], 'confidence': p['confidence'],
                 'latency_ms': p['latency_ms'], 'warning': p['warning']})
df = pd.DataFrame(rows)
print(json.dumps(summarize_metrics(rows), indent=2))
df.head(10)